# 10. De-identification scan and data release export

| | |
|---|---|
| **Stage** | 10 of 9 |
| **Purpose** | Scan every shareable data file for personal-data patterns and, in export mode, write a de-identified copy of the reviews, sentences, labels and predictions for archiving. |
| **Inputs** | `data/processed/{game}_clean.csv, {game}_sentences.parquet, {game}_sentences_weak.parquet`<br>`data/annotation_v2/gold_set_combined.csv, outputs/sentiment_audit.csv`<br>`outputs/predictions/{game}_aspects.parquet, {game}_pairs.parquet`<br>`outputs/dashboard_v4/*.parquet`<br>`outputs/supplementary/ (S5, IAA and adjudication files)` |
| **Outputs** | `release_data/ (same layout as the project folder, de-identified)`<br>`release_data/release_manifest.csv (export mode only)` |
| **Runtime** | CPU only. |
| **Depends on** | Stages 1 to 9 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

## What this notebook does

The saved review files contain user names, and the review text can contain personal data that players typed themselves (game ids, e-mail addresses, social handles, phone-like numbers). This notebook finds such patterns and, when asked, writes a de-identified copy of the data for archiving or sharing. The original files are never modified.

**Two modes** (set `MODE` in the cell):
- `scan` (default): read-only. Counts, per file and per pattern, the rows that match. It prints counts only, never the matched text.
- `export`: writes copies to `GameSense/release_data/` with the same folder layout, without the `userName` and `userImage` columns, and with matches replaced by the placeholders `[EMAIL]`, `[URL]`, `[HANDLE]`, `[NUMBER]` and `[PHONE]`. A manifest with row counts, sizes and checksums is written next to the files.

**How to use it**
1. Run with `MODE = "scan"` and read the counts. Email and link matches in the review files come mostly from developer replies (support addresses), not from players.
2. Run with `MODE = "export"`.
3. Store the result where access can be controlled (a private repository or a shared drive folder), not in a public repository.

**Notes**
- The patterns are conservative and also match harmless text, such as a long number that is not an id; the placeholders then replace that text.
- The exported text differs from the working files in the redacted places only. The models were trained on the working files, so a handful of sentences differ slightly.
- This notebook does not decide what may be shared. Reviews are public Google Play text, but redistributing a large collection of them can be restricted by the store's terms; check before publishing.

In [ ]:
# ===== De-identification scan and archive export for the review data =====
# MODE = "scan"   : READ-ONLY. Counts personal-data patterns in every file that could be shared. Prints counts only, never the matched text.
# MODE = "export" : writes redacted copies (no userName, patterns replaced by placeholders) to GameSense/release_data/. Originals are never modified.
import os, re, glob, hashlib, json
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

MODE = "scan"                       # "scan" first; switch to "export" after reading the scan
ROOT = "/content/drive/MyDrive/GameSense"
OUT = f"{ROOT}/release_data"
GAMES = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
DROP_COLS = {"userName", "userImage"}                                          # removed from every exported file
TEXT_COLS = {"content", "content_clean", "replyContent", "sentence_text", "review_text"}   # scanned and redacted

# (name, placeholder, regex); applied in this order
PATTERNS = [
    ("email",       "[EMAIL]",  re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")),
    ("url",         "[URL]",    re.compile(r"https?://\S+|www\.\S+")),
    ("handle",      "[HANDLE]", re.compile(r"(?<![\w@])@\w{3,}")),
    ("long_number", "[NUMBER]", re.compile(r"(?<!\d)\d{8,}(?!\d)")),                         # game / account ids are often 8 or more digits
    ("phone_like",  "[PHONE]",  re.compile(r"(?<!\d)\+?\d[\d\s().-]{7,}\d(?!\d)")),          # digits with separators
]

# files to scan or export: (path relative to ROOT, kind)
FILES = [(f"data/processed/{g}_clean.csv", "csv") for g in GAMES] + [(f"data/processed/{g}_{k}.parquet", "parquet") for g in GAMES for k in ("sentences", "sentences_weak")]
FILES += [("data/annotation_v2/gold_set_combined.csv", "csv"), ("outputs/sentiment_audit.csv", "csv")]
FILES += [(f"outputs/predictions/{g}_{k}.parquet", "parquet") for g in GAMES for k in ("aspects", "pairs")]
FILES += [(f"outputs/dashboard_v4/{n}.parquet", "parquet") for n in ("pairs", "examples", "reviews_meta")]            # what the public dashboard repository contains
FILES += [(f"outputs/supplementary/{n}", "csv") for n in ("s5_annotated.csv", "s5_key_DO_NOT_OPEN_BEFORE_ANNOTATING.csv", "s5_verify_annotated_by_annotator2.csv",
                                                           "s5_adjudicated.csv", "iaa_disagreements.csv", "iaa_annotator2.csv", "iaa_annotator2_topup.csv", "gold_corrections.csv")]

def redact_series(s):
    s = s.astype("string"); counts = {n: 0 for n, _, _ in PATTERNS}
    for name, ph, rx in PATTERNS:
        counts[name] = int(s.str.count(rx).fillna(0).sum()); s = s.str.replace(rx, ph, regex=True)
    return s, counts
def count_series(s):
    s = s.astype("string")
    return {n: int(s.str.contains(rx).fillna(False).sum()) for n, _, rx in PATTERNS}          # number of rows with at least one match

def read(path, kind):
    return pd.read_csv(path, low_memory=False) if kind == "csv" else pd.read_parquet(path)

rows, manifest = [], []
for rel, kind in FILES:
    p = f"{ROOT}/{rel}"
    if not os.path.exists(p):
        continue
    df = read(p, kind)
    cols = [c for c in df.columns if c in TEXT_COLS]
    has_user = [c for c in df.columns if c in DROP_COLS]
    row = {"file": rel, "rows": len(df), "user-name columns": ",".join(has_user) or "-", "text columns scanned": ",".join(cols) or "-"}
    tot = {n: 0 for n, _, _ in PATTERNS}; red_rows = 0
    out = df.drop(columns=has_user)
    for c in cols:
        if MODE == "scan":
            for n, v in count_series(df[c]).items(): tot[n] += v
        else:
            new, cnt = redact_series(df[c])
            red_rows += int((new.fillna("") != df[c].astype("string").fillna("")).sum()); out[c] = new
            for n, v in cnt.items(): tot[n] += v
    row.update({f"{n} (rows)" if MODE == "scan" else f"{n} (matches)": v for n, v in tot.items()})
    if MODE == "export":
        row["rows changed"] = red_rows
        dst = f"{OUT}/{rel}"; os.makedirs(os.path.dirname(dst), exist_ok=True)
        out.to_csv(dst, index=False) if kind == "csv" else out.to_parquet(dst, index=False)
        manifest.append({"file": rel, "rows": len(out), "columns": len(out.columns), "size_MB": round(os.path.getsize(dst) / 1024 / 1024, 2),
                         "sha256_12": hashlib.sha256(open(dst, "rb").read()).hexdigest()[:12], "dropped columns": ",".join(has_user) or "-"})
    rows.append(row)

res = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
print(f"MODE = {MODE}\n")
print(res.to_string(index=False))
if MODE == "scan":
    num = [c for c in res.columns if c.endswith("(rows)")]
    print("\nTOTAL rows with at least one match, per pattern:", res[num].sum().to_dict())
    print("Files that contain user-name columns:", res.loc[res["user-name columns"] != "-", "file"].tolist())
    print("\nCounts are rows with at least one match, summed over the scanned text columns (a row can match several patterns, and the same text in")
    print("content and content_clean is counted twice). The patterns are conservative and may also match harmless text (for example a long number that is not an id).")
    print("Nothing was written. Set MODE = 'export' to write redacted copies to", OUT)
else:
    pd.DataFrame(manifest).to_csv(f"{OUT}/release_manifest.csv", index=False)
    print(f"\nExported {len(manifest)} files, {sum(m['size_MB'] for m in manifest):.1f} MB, to {OUT}\nmanifest: {OUT}/release_manifest.csv")
